# Split - Apply - Combine strategy

![](https://img1.daumcdn.net/thumb/R800x0/?scode=mtistory2&fname=https%3A%2F%2Ft1.daumcdn.net%2Fcfile%2Ftistory%2F9978503F5B8264490F)

![](https://t1.daumcdn.net/cfile/tistory/9961104E5B8BB34A22)


In [ ]:
# '데이터' 관련 하여 어떠한 문제 해결에 있어서

# split - apply - combine  전략 사용.
# (분할 - 적용 - 병합)

#  'R' 에서도 이와 같이 구현되었는데  (
#      그룹으로 나누고 (split)
#      동일한 연산 수행후에 (apply)
#      결과를 합치는 과정 (combine)

# 원래 R 에서 구현하던 방식. 
# 이걸 고안하신 Hadley Wickham (R 생태계 정립하신분)의 논문 "Strategy of Split-Apply-Combine" 에서.  
# 꽤 많은 문제들이 Split-Apply-Combine 으로 '문제해결' 을 할수 있슴을 주장
# https://www.jstatsoft.org/v40/i01/

# 엑셀등에서도 많이 하는 작업

# 가령 
#     ex) '일별'로 묶어서 일별 매출 구하기
#     ex) '상품 카테고리별'로 묶어서 상품이 특정 기간의 매출 구하기
#     ex) '유저' 별 매출


# 여기서 복잡해 지는 상황은 바로 'apply과정'입니다

# 'apply 과정'에서 적용할수 있는 대표적인 함수들
#    - aggregate() agg() : N => 1    집계하는 연산, 'N개를 입력' 받아서 '1개의 결과'를 만들어 내기    ex)합계
#    - transform() : N => N         'N개를 입력'받아서  '전체' 를 바꿈
#    - apply() : N => 1, N, M...   'N개를 입력'받아서  '1개' 'N개' 혹은 'M개' 의 결과 를 만들어 냄.
#    - filter() : N => <=N  N보다 같거나 적은 결과


# ※ apply() 만으로도 모~든 것들이 가능하기도 하나. 각각의 데이터 연산 동작에 따른 함수를 사용하여 'apply' 하는게 좋다.


# agg, transform, apply, filter 요약비교

| 함수                           | 주요 목적                           | 반환 형태                              | 사용 맥락 | 핵심 특징                                          |
| ---------------------------- | ------------------------------- | ---------------------------------- | ----- | ---------------------------------------------- |
| **aggregate()** (또는 `agg()`) | 그룹별 **요약 통계** 계산                | **줄어든 DataFrame / Series**         | 집계    | 여러 함수 적용 가능 (`sum`, `mean`, `['sum', 'mean']`) |
| **transform()**              | 그룹별 계산 결과를 **원래 크기 그대로 반환**     | **원래와 동일한 길이의 Series / DataFrame** | 전처리   | 각 그룹에 함수 적용 후 원래 인덱스 유지                        |
| **apply()**                  | 그룹별로 **임의의 연산**을 적용             | **자유로운 반환 형태**                     | 범용    | 집계, 변형, 복잡한 연산 모두 가능                           |
| **filter()**                 | 그룹을 조건에 따라 **걸러냄 (True/False)** | **줄어든 DataFrame**                  | 선택    | 조건에 맞는 그룹만 남김                                  |



**비유**
| 함수            | 비유                              |
| ------------- | ------------------------------- |
| `aggregate()` | “각 반의 평균 점수표 만들기”               |
| `transform()` | “학생별로 자기 반의 평균 점수 옆에 적기”        |
| `apply()`     | “각 반별로 복잡한 연산(정렬, 커스텀 계산 등) 수행” |
| `filter()`    | “평균이 50점 이상인 반만 남기기”            |



# import & 기본데이터 & groupby 준비

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
data = {
    '팀': ['A', 'A', 'A', 'B', 'B', 'C', 'C', 'C'],
    '선수': ['철수', '영희', '민수', '지영', '현수', '수진', '태민', '지훈'],
    '점수': [10, 20, 30, 40, 25, 5, 15, 10],
    '어시스트': [3, 7, 4, 8, 6, 2, 1, 3]
}
df = pd.DataFrame(data)
df


,팀,선수,점수,어시스트
0,A,철수,10,3
1,A,영희,20,7
2,A,민수,30,4
3,B,지영,40,8
4,B,현수,25,6
5,C,수진,5,2
6,C,태민,15,1
7,C,지훈,10,3


In [3]:
group = df.groupby('팀')

group.size()

팀
A    3
B    2
C    3
dtype: int64

In [4]:
# 각 그룹의 DataFrame
group.get_group('A')

,팀,선수,점수,어시스트
0,A,철수,10,3
1,A,영희,20,7
2,A,민수,30,4


In [5]:
group.get_group('B')

,팀,선수,점수,어시스트
3,B,지영,40,8
4,B,현수,25,6


In [6]:
group.get_group('C')

,팀,선수,점수,어시스트
5,C,수진,5,2
6,C,태민,15,1
7,C,지훈,10,3


In [7]:
group['점수']  # SeriesGroupBy

In [8]:
group[['점수']]  # DataFrameGroupBy

In [11]:
group[['점수']].get_group('A')  # 결과 DataFrame

,점수
0,10
1,20
2,30


In [12]:
group['점수'].get_group('A')  # 결과 Series

0    10
1    20
2    30
Name: 점수, dtype: int64

# 🔶 agg(), aggregate() 
- 그룹별 집계 (평균, 합계, 개수, 최대값..)
- 그룹 크키가 줄어듦 (그룹당 1행씩)
- 활용예) “각 반의 평균 점수표 만들기”

In [14]:
group[['점수']].agg('mean')  # DataFrameGroupBy 객체에 agg(단일집계함수) => DataFrame

,점수
팀,
A,20.0
B,32.5
C,10.0


In [15]:
group[['점수', '어시스트']].agg('mean')

,점수,어시스트
팀,,
A,20.0,4.666667
B,32.5,7.000000
C,10.0,2.000000


In [16]:
group['점수'].agg('mean')  # SeriesGroupBy 객체에 agg(단일집계함수) => Series 로 combine

팀
A    20.0
B    32.5
C    10.0
Name: 점수, dtype: float64

In [18]:
group['점수'].agg(['mean'])  # SeriesGroupBy 객체에 agg(단일집계함수들) => DataFrame

,mean
팀,
A,20.0
B,32.5
C,10.0


In [19]:
group['점수'].agg(['mean', 'sum', 'max']) 

,mean,sum,max
팀,,,
A,20.0,60,30
B,32.5,65,40
C,10.0,30,15


In [24]:
# agg(dict)
group[df.columns].agg({
    '점수': ['sum', 'mean'],
    '어시스트': ['count'],
    '선수': lambda x: ','.join(x),   # x 는 각 그룹의 '선수'컬럼
})


점수        어시스트        선수
  sum  mean count  <lambda>
팀                          
A  60  20.0     3  철수,영희,민수
B  65  32.5     2     지영,현수
C  30  10.0     3  수진,태민,지훈

In [28]:
group[df.columns].agg({
    '점수': ['sum', 'mean'],
    '어시스트': ['count'],
    '선수': lambda x: ','.join(x),   # x 는 각 그룹의 '선수'컬럼
})\
.rename(columns={
    'sum': '점수합계',
    'mean': '점수평균',
    'count': '어시스트개수',
    '<lambda>': '선수들'
})

점수         어시스트        선수
  점수합계  점수평균 어시스트개수       선수들
팀                            
A   60  20.0      3  철수,영희,민수
B   65  32.5      2     지영,현수
C   30  10.0      3  수진,태민,지훈

In [29]:
group[df.columns].agg({
    '점수': ['sum', 'mean'],
    '어시스트': ['count'],
    '선수': lambda x: ','.join(x),   # x 는 각 그룹의 '선수'컬럼
})\
.rename(columns={
    'sum': '점수합계',
    'mean': '점수평균',
    'count': '어시스트개수',
    '<lambda>': '선수들'
})\
.droplevel(0, axis=1)

,점수합계,점수평균,어시스트개수,선수들
팀,,,,
A,60,20.0,3,"철수,영희,민수"
B,65,32.5,2,"지영,현수"
C,30,10.0,3,"수진,태민,지훈"


## Named Aggregation 방식추천
- `<lambda>` 문자열을 rename에 넣는 방식은 두 가지 이유로 껄끄러운데, 
하나는 가독성이고 하나는 안정성이이다
(람다가 여러 개면 `<lambda_0>`, `<lambda_1>`처럼 인덱스가 붙어서 깨지기 쉽습니다).

- pandas의 named aggregation 문법을 쓰면 애초에 MultiIndex 컬럼이 생기지 않고, 원하는 이름을 바로 지정할 수 있어서 rename() 자체가 필요 없어집니다


In [32]:
group.agg(
    점수합계=('점수', 'sum'),
    점수평균=('점수', 'mean'),
    어시스트=('어시스트', 'count'),
    선수들=('선수', lambda x: ','.join(x)),
)

,점수합계,점수평균,어시스트,선수들
팀,,,,
A,60,20.0,3,"철수,영희,민수"
B,65,32.5,2,"지영,현수"
C,30,10.0,3,"수진,태민,지훈"


# 🔶 transform()
- 그룹별 계산후 원래 형태 유지
- 머신러닝, 데이터전처리에서 그룹별 표준화, 그룹평균대체 등에 활용됨
- 활용예) “학생별로 자기 반의 평균 점수 옆에 적기”


In [34]:
group['점수'].agg('mean')

팀
A    20.0
B    32.5
C    10.0
Name: 점수, dtype: float64

In [37]:
group['점수'].transform('mean')
# 그룹 각 행에 그 그룹의 '점수' 평균을 넣어줌. 
# 원래 행수가 유지됨.      

0    20.0
1    20.0
2    20.0
3    32.5
4    32.5
5    10.0
6    10.0
7    10.0
Name: 점수, dtype: float64

In [39]:
df['팀별_평균점수'] = group['점수'].transform('mean')
df

,팀,선수,점수,어시스트,팀별_평균점수
0,A,철수,10,3,20.0
1,A,영희,20,7,20.0
2,A,민수,30,4,20.0
3,B,지영,40,8,32.5
4,B,현수,25,6,32.5
5,C,수진,5,2,10.0
6,C,태민,15,1,10.0
7,C,지훈,10,3,10.0


In [40]:
# transform 활용
df['평균대비점수'] = df['점수'] - df['팀별_평균점수']
df

,팀,선수,점수,어시스트,팀별_평균점수,평균대비점수
0,A,철수,10,3,20.0,-10.0
1,A,영희,20,7,20.0,0.0
2,A,민수,30,4,20.0,10.0
3,B,지영,40,8,32.5,7.5
4,B,현수,25,6,32.5,-7.5
5,C,수진,5,2,10.0,-5.0
6,C,태민,15,1,10.0,5.0
7,C,지훈,10,3,10.0,0.0


# 🔶 filter()
- 조건에 맞는 그룹만 남기기. 그룹 필터링
- 그룹 단위로 조건을 검사한 뒤 → True인 그룹만 전체 데이터프레임에서 유지
- 활용예 “평균이 50점 이상인 반만 남기기”


In [41]:
# group: DataFrameGroupby 객체
#  따라서 x는 각 그룹의 DataFrame 객체
group.filter(lambda x: x['점수'].mean() >= 15)

,팀,선수,점수,어시스트,팀별_평균점수,평균대비점수
0,A,철수,10,3,20.0,-10.0
1,A,영희,20,7,20.0,0.0
2,A,민수,30,4,20.0,10.0
3,B,지영,40,8,32.5,7.5
4,B,현수,25,6,32.5,-7.5


# 🔶 apply()
- 자유도 높은 적용 (복잡한 연산 가능)
- 리턴형태가 자유로움 (Series, DataFrame, scalar 등)

In [42]:
# pandas의 apply() 는 여러 객체(Series, DataFrame, GroupBy, Rolling, Expanding, Resampler)에서 각각 다르게 동작합니다. 

## pandas 에서 apply() 를 사용할수 있는 주요 객체들

- Series.apply(func)          → 각 원소 단위로 적용
- DataFrame.apply(func, axis) → 행 또는 열 단위로 적용
- GroupBy.apply(func)         → 그룹 단위로 적용
- Rolling.apply(func)         → 윈도우 단위로 적용
- Expanding.apply(func)       → 누적 단위로 적용
- Resampler.apply(func)       → 시계열 리샘플링 단위로 적용


| 객체                                        | 입력 함수의 인자 (입력 형태)                        | 출력 형태                             | 설명 / 특징                                  | 예시 코드                                        |
| ----------------------------------------- | ---------------------------------------- | --------------------------------- | ---------------------------------------- | -------------------------------------------- |
| **`Series`**                              | 각 원소 (`scalar`)                          | `scalar`, `Series`, `list`, `any` | Series의 각 원소에 함수를 적용. Python `map()`과 유사 | `s.apply(lambda x: x**2)`                    |
| **`DataFrame`**                           | 각 **행(row)** 또는 **열(column)** (`Series`) | `Series`, `DataFrame`, `scalar`   | `axis=0` → 열 기준, `axis=1` → 행 기준         | `df.apply(np.sum, axis=0)`                   |
| **`GroupBy` 객체 (`df.groupby('col')`)**    | 각 그룹별 **DataFrame 또는 Series**            | `Series` 또는 `DataFrame`           | 그룹별 사용자 정의 함수를 적용 (group 단위 처리)          | `df.groupby('key').apply(lambda g: g.sum())` |
| **`Rolling` 객체 (`df.rolling(window=3)`)** | 각 구간(window)별 **Series**                 | `Series`                          | 이동(window) 단위 계산 (시계열 등에서 주로 사용)         | `df['A'].rolling(3).apply(np.mean)`          |
| **`Expanding` 객체 (`df.expanding()`)**     | 누적(expanding) 구간의 **Series**             | `Series`                          | 누적 계산용, 처음부터 현재까지 누적 구간 단위 적용            | `df['A'].expanding().apply(np.std)`          |
| **`Resampler` 객체 (`df.resample('M')`)**   | 리샘플된 각 구간의 **DataFrame**                 | `Series` 또는 `DataFrame`           | 시계열 데이터를 주기별로 나누어 적용                     | `df.resample('M').apply(np.mean)`            |



### 일반적으로 추천하는 상황
| 상황          | 추천 메서드                                  | 이유            |
| ----------- | --------------------------------------- | ------------- |
| 원소별 연산      | `Series.apply()` 또는 `map()`             | 빠르고 간단        |
| 행·열 단위 계산   | `DataFrame.apply()`                     | `axis`로 제어 가능 |
| 그룹별 연산      | `GroupBy.apply()`                       | 그룹 단위로 자유도 높음 |
| 이동 평균/누적 계산 | `Rolling.apply()` / `Expanding.apply()` | 시계열 분석용       |
| 리샘플링 주기별 계산 | `Resampler.apply()`                     | 예: 월별/주별 통계   |




In [43]:
s = pd.Series([1, 2, 3])
print(s)

0    1
1    2
2    3
dtype: int64


## 1️⃣ Series.apply()

- apply() 결과 : `pd.Series` 또는 `pd.DataFrame`


- 내부함수
```python
def func(x: Any) -> Any:
    return x ** 2

s.apply(func)  # 입력: 각 원소(scalar)
```

* **입력 타입:** `Any` (ex: `int`, `float`, `str`)
* **출력 타입:** `Any` (단일 값 또는 시퀀스)
* **적용 대상:** Series의 각 원소


In [44]:
s.apply(lambda x: type(x))  # 단일값 리턴하는 함수 -> apply 결과는 Series

0    <class 'int'>
1    <class 'int'>
2    <class 'int'>
dtype: object

In [45]:
s.apply(lambda x: x ** 2)

0    1
1    4
2    9
dtype: int64

In [47]:
# 시퀀스 리턴하는 함수 -> apply 결과는 DataFrame
s.apply(lambda x: pd.Series([x, x ** 2, x ** 3], ['x', 'x^2', 'x^3']))

,x,x^2,x^3
0,1,1,1
1,2,4,8
2,3,9,27


## 2️⃣ DataFrame.apply()

- apply() 결과 : `pd.Series` 또는 `pd.DataFrame`

- 내부함수
```python
def func(col: pd.Series) -> Any:
    return col.sum()

df.apply(func, axis=0)  # 각 열(Series)이 입력됨
```

* **입력 타입:** `pd.Series`
* **출력 타입:** `Any` (`scalar`, `pd.Series`, `pd.DataFrame`)
* **적용 대상:** `axis=0` → 각 열, `axis=1` → 각 행

예시:

```python
def row_func(row: pd.Series) -> float:
    return row['A'] + row['B']

df.apply(row_func, axis=1)
```


In [48]:
df

,팀,선수,점수,어시스트,팀별_평균점수,평균대비점수
0,A,철수,10,3,20.0,-10.0
1,A,영희,20,7,20.0,0.0
2,A,민수,30,4,20.0,10.0
3,B,지영,40,8,32.5,7.5
4,B,현수,25,6,32.5,-7.5
5,C,수진,5,2,10.0,-5.0
6,C,태민,15,1,10.0,5.0
7,C,지훈,10,3,10.0,0.0


In [50]:
df.apply(lambda x: type(x))  # x는 '열'의 Series (axis=0, 디폴트)

팀          <class 'pandas.Series'>
선수         <class 'pandas.Series'>
점수         <class 'pandas.Series'>
어시스트       <class 'pandas.Series'>
팀별_평균점수    <class 'pandas.Series'>
평균대비점수     <class 'pandas.Series'>
dtype: object

In [49]:
df.apply(lambda x: type(x), axis=1)  # x는 '행'의 Series

0    <class 'pandas.Series'>
1    <class 'pandas.Series'>
2    <class 'pandas.Series'>
3    <class 'pandas.Series'>
4    <class 'pandas.Series'>
5    <class 'pandas.Series'>
6    <class 'pandas.Series'>
7    <class 'pandas.Series'>
dtype: object

In [ ]:
df.apply(np.sum, axis=0)  # 열 별로 sum 결과

In [51]:
df.apply(np.sum, axis=1)

TypeError: can only concatenate str (not "int") to str

In [52]:
df[['점수', '어시스트']].apply(np.sum, axis=1)

0    13
1    27
2    34
3    48
4    31
5     7
6    16
7    13
dtype: int64

In [54]:
df.apply(lambda row: row['점수'] + row['어시스트'], axis=1)

0    13
1    27
2    34
3    48
4    31
5     7
6    16
7    13
dtype: int64

## 3️⃣ GroupBy.apply()
- recap: groupby() 결과는 DataFrameGroupBy 이나 SeriesGroupBy 일수 있다

- apply() 결과 : `pd.Series` 또는 `pd.DataFrame`

- 내부함수
```python
def func(group: pd.DataFrame) -> pd.DataFrame:
    return group[['A', 'B']].sum()

df.groupby('key').apply(func)
```

* **입력 타입:** `pd.Series` 또는 `pd.DataFrame` (← 그룹별 데이터)
* **출력 타입:** `pd.Series` 또는 `pd.DataFrame`
* **적용 대상:**  각 그룹 전체


In [56]:
df

,팀,선수,점수,어시스트,팀별_평균점수,평균대비점수
0,A,철수,10,3,20.0,-10.0
1,A,영희,20,7,20.0,0.0
2,A,민수,30,4,20.0,10.0
3,B,지영,40,8,32.5,7.5
4,B,현수,25,6,32.5,-7.5
5,C,수진,5,2,10.0,-5.0
6,C,태민,15,1,10.0,5.0
7,C,지훈,10,3,10.0,0.0


In [57]:
group.size()

팀
A    3
B    2
C    3
dtype: int64

In [58]:
group[['선수']].apply(lambda g: type(g))

팀
A    <class 'pandas.DataFrame'>
B    <class 'pandas.DataFrame'>
C    <class 'pandas.DataFrame'>
dtype: object

In [60]:
group[['선수']].apply(lambda g: g.shape)

팀
A    (3, 1)
B    (2, 1)
C    (3, 1)
dtype: object

In [61]:
group['선수'].apply(lambda g: g.shape)

팀
A    (3,)
B    (2,)
C    (3,)
Name: 선수, dtype: object

In [62]:
group.apply(lambda g: g.shape)

팀
A    (3, 5)
B    (2, 5)
C    (3, 5)
dtype: object

In [63]:
df.columns

Index(['팀', '선수', '점수', '어시스트', '팀별_평균점수', '평균대비점수'], dtype='str')

In [64]:
group[df.columns].apply(lambda g: pd.Series([10, 20, 30]))

,0,1,2
팀,,,
A,10,20,30
B,10,20,30
C,10,20,30


In [65]:
group[df.columns].apply(lambda g: g[['어시스트', '선수']])

어시스트  선수
팀            
A 0     3  철수
  1     7  영희
  2     4  민수
B 3     8  지영
  4     6  현수
C 5     2  수진
  6     1  태민
  7     3  지훈

In [66]:
group[df.columns].apply(lambda g: g[['어시스트', '선수']]).droplevel(-1, axis=0)

,어시스트,선수
팀,,
A,3,철수
A,7,영희
A,4,민수
B,8,지영
B,6,현수
C,2,수진
C,1,태민
C,3,지훈
